# Silver | Qualidade e integridade referencial
Executar depois dos tratamentos e dimensões da Silver.

| Regra | Severidade |
|---|---|
| `CHAVE_NULA`, `CHAVE_DUPLICADA` (todas as tabelas Silver) | BLOQUEANTE |
| `TAXA_CORRESPONDENCIA_JOIN` ≥ limiar (município×meta, alunos×município, UF×meta) | BLOQUEANTE |
| `CODIGO_DESCONHECIDO` (valor categórico fora do dicionário oficial) | BLOQUEANTE (INFORMATIVA se o dicionário não cobre a coluna) |
| `RELACIONAMENTO_SEM_*` (contagem de órfãos para investigação) | INFORMATIVA |

**Gate:** todas as regras são persistidas em `workspace.monitoring.qualidade_resultados` **antes** de a exceção ser levantada,
e a falha também é registrada em `pipeline_runs` (status `FALHA`) antes de ser propagada com `raise`.

**Comparações resultado × meta só valem no mesmo recorte:** a meta municipal existe só para a rede municipal (código '3');
a meta estadual é um agregado da rede **Pública**. Por isso os joins abaixo restringem o lado esquerdo ao recorte correspondente.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
dbutils.widgets.text("limiar_correspondencia", "95", "Limiar mínimo de correspondência (%)")
LIMIAR = float(dbutils.widgets.get("limiar_correspondencia"))

_m = iniciar_etapa()
dq = RegistroDQ("SILVER", limiar_correspondencia=LIMIAR)

In [0]:
# 1. Chaves nulas e duplicadas (bloqueantes)
CHECKS = [
    ("uf", ["ano", "sigla_uf", "serie", "rede_codigo"]),
    ("meta_alfabetizacao_brasil", ["ano", "rede_codigo"]),
    ("meta_alfabetizacao_uf", ["ano", "sigla_uf", "rede_codigo"]),
    ("meta_alfabetizacao_municipio", ["ano", "id_municipio"]),
    ("municipio", ["ano", "id_municipio", "serie", "rede_codigo"]),
    ("alunos", ["ano", "id_municipio", "id_escola", "id_aluno"]),
    ("diretorio_municipio", ["id_municipio"]),
    ("diretorio_uf", ["id_uf"]),
]
for tabela, chaves in CHECKS:
    dq.chave_nula_e_duplicada(f"workspace.silver.{tabela}", chaves)

In [0]:
# 2. Códigos categóricos x dicionário oficial (sem fallback: valor novo na fonte é detectado)
# (tabela, coluna na Silver, nome da coluna no dicionário)
CATEGORICAS = [
    ("uf", "rede_codigo", "rede"), ("uf", "serie", "serie"),
    ("municipio", "rede_codigo", "rede"), ("municipio", "serie", "serie"),
    ("alunos", "rede_codigo", "rede"), ("alunos", "serie", "serie"),
    ("alunos", "presenca", "presenca"), ("alunos", "alfabetizado", "alfabetizado"),
]
for tabela, coluna, col_dic in CATEGORICAS:
    n_dic = spark.sql(f"""
        SELECT COUNT(*) n FROM workspace.silver.dicionario
        WHERE id_tabela = '{tabela}' AND nome_coluna = '{col_dic}'""").first()["n"]
    if n_dic == 0:
        dq.registrar(f"workspace.silver.{tabela}", f"CODIGO_DESCONHECIDO_{coluna.upper()}", "INFORMATIVA", 0,
                     f"Regra não aplicada: dicionário não tem entradas para {tabela}.{col_dic}")
        continue
    desconhecidos = spark.sql(f"""
        SELECT COUNT(*) n FROM workspace.silver.{tabela} t
        LEFT ANTI JOIN (SELECT CAST(chave AS STRING) AS chave FROM workspace.silver.dicionario
                        WHERE id_tabela = '{tabela}' AND nome_coluna = '{col_dic}') d
          ON CAST(t.`{coluna}` AS STRING) = d.chave
        WHERE t.`{coluna}` IS NOT NULL""").first()["n"]
    dq.registrar(f"workspace.silver.{tabela}", f"CODIGO_DESCONHECIDO_{coluna.upper()}", "BLOQUEANTE",
                 desconhecidos, f"Valores de {coluna} ausentes do dicionário ({tabela}.{col_dic}) — contagem de linhas")

In [0]:
# 3. Taxa de correspondência dos joins (bloqueante)
REDE_PUBLICA = """(SELECT DISTINCT CAST(chave AS STRING) AS rede_codigo
                   FROM workspace.silver.dicionario
                   WHERE nome_coluna = 'rede' AND TRIM(valor) LIKE 'Pública%')"""

# 3.1 município x meta municipal: a meta só cobre a rede municipal ('3')
taxa1 = dq.taxa_correspondencia(
    "municipio x meta_municipio",
    "(SELECT * FROM workspace.silver.municipio WHERE rede_codigo = '3')",
    "workspace.silver.meta_alfabetizacao_municipio",
    "l.ano = r.ano AND l.id_municipio = r.id_municipio AND l.rede_codigo = r.rede_codigo")

# 3.2 alunos x município (por ano e município distintos)
taxa2 = dq.taxa_correspondencia(
    "alunos x municipio",
    "(SELECT DISTINCT ano, id_municipio FROM workspace.silver.alunos)",
    "workspace.silver.municipio",
    "l.ano = r.ano AND l.id_municipio = r.id_municipio")

# 3.3 UF x meta estadual: a meta é agregado da rede PÚBLICA → só comparar resultados da rede pública
#     (comparar todas as redes com a meta pública seria resultado de uma rede x meta de outra)
taxa3 = dq.taxa_correspondencia(
    "uf (rede pública) x meta_uf",
    f"(SELECT u.* FROM workspace.silver.uf u LEFT SEMI JOIN {REDE_PUBLICA} p ON u.rede_codigo = p.rede_codigo)",
    "workspace.silver.meta_alfabetizacao_uf",
    "l.ano = r.ano AND l.sigla_uf = r.sigla_uf")

print(f"município×meta={taxa1:.2f}% | alunos×município={taxa2:.2f}% | uf×meta={taxa3:.2f}% (limiar {LIMIAR}%)")

In [0]:
# 4. Órfãos para investigação (informativas; não excluir registros nem presumir causa)
sem_meta = spark.sql("""
    SELECT COUNT(*) n FROM workspace.silver.municipio r
    LEFT ANTI JOIN workspace.silver.meta_alfabetizacao_municipio m
      ON r.ano = m.ano AND r.id_municipio = m.id_municipio AND r.rede_codigo = m.rede_codigo
    WHERE r.rede_codigo = '3'""").first()["n"]
dq.registrar("municipio x meta_alfabetizacao_municipio", "RELACIONAMENTO_SEM_META_REDE_MUNICIPAL",
             "INFORMATIVA", sem_meta,
             "Resultados da rede Municipal sem meta no mesmo ano/município/rede. Ausência de meta NÃO é meta zero.")

por_ano = spark.sql("""
    SELECT a.ano, COUNT(*) AS n FROM workspace.silver.alunos a
    LEFT ANTI JOIN (SELECT DISTINCT ano, id_municipio FROM workspace.silver.municipio) m
      ON a.ano = m.ano AND a.id_municipio = m.id_municipio
    GROUP BY a.ano ORDER BY a.ano""").collect()
dq.registrar("alunos x municipio", "RELACIONAMENTO_SEM_MUNICIPIO", "INFORMATIVA",
             sum(int(r["n"]) for r in por_ano),
             "Alunos sem município no mesmo ano. Por ano: " + "; ".join(f"{r['ano']}: {r['n']}" for r in por_ano))

In [0]:
# 5. Gate final: registra a falha em pipeline_runs e PROPAGA (sem except genérico que engula o erro)
try:
    dq.gate()
except Exception as exc:
    registrar_execucao("SILVER", "dq_silver", _m, status="FALHA", erro=exc)
    raise
registrar_execucao("SILVER", "dq_silver", _m, status="SUCESSO")